In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

COTAS_CM = list(range(300, 531, 10)) + [535]

candidatas = [Path.cwd(), Path.cwd().parent]
RAIZ = next((p for p in candidatas
             if (p / "dados/processados/api_casasbombaebap_convertido.json").is_file()
             and (p / "dados/originais/dadosapi_cotasinundacao.json").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Nao encontrei as duas entradas. Coloque o notebook na pasta do projeto, notebooks ou scripts.")
raiz = RAIZ.resolve()
caminho_casas = raiz / "dados/processados/api_casasbombaebap_convertido.json"
caminho_cotas = raiz / "dados/originais/dadosapi_cotasinundacao.json"
print("Entradas encontradas. Pronto para executar a limpeza.")


Entradas encontradas. Pronto para executar a limpeza.


In [2]:
def to_snake(name):
    """Padroniza apenas NOMES DE COLUNAS; preserva partes como cm e m."""
    name = unicodedata.normalize("NFKD", str(name))
    name = name.encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

def texto(valor):
    """Preserva acentos nos valores; espacos isolados viram ausencia."""
    if isinstance(valor, str):
        valor = unicodedata.normalize("NFC", " ".join(valor.split()))
        return valor if valor else None
    return valor

def ler_json(caminho):
    with open(caminho, encoding="utf-8") as arquivo:
        return json.load(arquivo)


In [3]:
def clean(raw):
    """P1-P4: limpa o cadastro convertido e registra as colunas retiradas."""
    if raw.get("error") or raw.get("exceededTransferLimit"):
        raise ValueError("JSON de casas com erro ou coleta incompleta.")
    referencia = raw.get("spatialReference", {})
    if referencia.get("latestWkid", referencia.get("wkid")) != 4326:
        raise ValueError("Use api_casasbombaebap_convertido.json em EPSG:4326.")

    # P1: transformar o JSON aninhado em tabela, uma linha por casa.
    linhas = []
    for casa in raw["features"]:
        linhas.append({**casa["attributes"],
                       "longitude": casa["geometry"]["x"],
                       "latitude": casa["geometry"]["y"]})
    df = pd.DataFrame(linhas)
    print(f"{len(df)} casas de bomba lidas")
    df = df.rename(columns=to_snake)
    if df.columns.duplicated().any():
        raise ValueError("Padronizacao gerou nomes de colunas repetidos.")

    # P2: tratar strings vazias antes de procurar colunas totalmente vazias.
    for coluna in df.columns:
        if df[coluna].dtype == "object":
            df[coluna] = df[coluna].map(texto)
    vazias = df.columns[df.isna().all()].tolist()
    df = df.dropna(axis=1, how="all")

    # P3: selecionar atributos que respondem a pergunta da Tela 2.
    # Diferentemente do exemplo industrial, NAO removemos o identificador.
    necessarias = ["fid", "name", "longitude", "latitude"]
    faltantes = sorted(set(necessarias) - set(df.columns))
    if faltantes:
        raise ValueError(f"Colunas obrigatorias ausentes: {faltantes}")
    outras = [coluna for coluna in df.columns if coluna not in necessarias]
    df = df[necessarias].rename(columns={"name": "nome"}).copy()

    # P4: remover somente duplicatas identicas e conferir tipos/chaves.
    antes = len(df)
    df = df.drop_duplicates()
    duplicatas = antes - len(df)
    if df["fid"].map(lambda v: isinstance(v, bool)).any():
        raise ValueError("fid booleano invalido.")
    df["fid"] = pd.to_numeric(df["fid"], errors="raise")
    if df["fid"].isna().any() or (df["fid"] % 1 != 0).any():
        raise ValueError("fid deve ser inteiro e preenchido.")
    df["fid"] = df["fid"].astype("int64")
    for coluna in ["longitude", "latitude"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="raise")
    if df.isna().any().any():
        raise ValueError("Cadastro com identificador, nome ou coordenada ausente.")
    if df["fid"].duplicated().any() or df["nome"].duplicated().any():
        raise ValueError("Identificadores ou nomes repetidos com dados conflitantes.")
    # Faixa ampla de plausibilidade regional, nao limite municipal.
    if not (df["longitude"].between(-52, -50) & df["latitude"].between(-31, -29)).all():
        raise ValueError("Coordenadas fora da faixa esperada para a regiao.")
    return df.sort_values("fid"), {
        "colunas_totalmente_vazias": vazias,
        "outras_colunas_fora_da_tabela_analitica": outras,
        "duplicatas_identicas_removidas_do_cadastro_analitico": duplicatas,
    }


In [4]:
casas, auditoria = clean(ler_json(caminho_casas))
print("Colunas totalmente vazias retiradas:", len(auditoria["colunas_totalmente_vazias"]))
print(", ".join(auditoria["colunas_totalmente_vazias"]))
casas


24 casas de bomba lidas
Colunas totalmente vazias retiradas: 0



,fid,nome,longitude,latitude
0,1,CB-2,-51.214362,-30.016007
1,2,CB-3,-51.210352,-30.008697
2,3,CB-4,-51.207673,-30.002063
3,4,CB-5,-51.201240,-29.982009
4,5,CB-6,-51.175796,-29.969781
5,6,CB-9,-51.138591,-29.968089
6,7,CB-11B,-51.242660,-30.092271
7,8,CB-12,-51.236010,-30.061403
8,9,CB-13,-51.232040,-30.054590
9,10,CB-14,-51.215225,-30.048152


In [6]:
def clean_cotas(raw, casas):
    """P5-P7: limpa as consultas, cruza por fid e preserva as lacunas."""
    # A lista de consultas ja foi montada pelo notebook do grupo.
    # Verificamos repeticoes do registro completo antes de abrir os poligonos.
    unicas = {}
    duplicatas_consultas = 0
    for registro in raw:
        chave = registro["fid"]
        if isinstance(chave, bool) or not isinstance(chave, int):
            raise ValueError("fid das consultas deve ser inteiro.")
        if chave in unicas:
            if registro != unicas[chave]:
                raise ValueError(f"Consultas conflitantes para fid {chave}.")
            duplicatas_consultas += 1
        unicas[chave] = registro
    raw = list(unicas.values())
    if set(unicas) != set(casas["fid"]):
        raise ValueError("As consultas e o cadastro nao tem os mesmos fids.")

    # P5: conferir a identidade e as coordenadas usadas nas consultas espaciais.
    consulta_df = pd.DataFrame(raw)
    consulta_df["nome"] = consulta_df["nome"].map(texto)
    conferido = casas.merge(consulta_df, on="fid", validate="one_to_one",
                           suffixes=("", "_consulta"))
    if not conferido["nome"].eq(conferido["nome_consulta"]).all():
        raise ValueError("Nome da consulta diverge do cadastro.")
    for coluna in ["longitude", "latitude"]:
        outra = pd.to_numeric(conferido[coluna + "_consulta"], errors="raise")
        if not conferido[coluna].sub(outra).abs().le(1e-7).all():
            raise ValueError(f"{coluna} da consulta diverge do cadastro.")

    # P6: uma linha por casa e poligono; None nao vira zero.
    linhas = [{"fid": r["fid"], "objectid": p["objectid"], "cota_cm": p["contourmax"]}
              for r in raw for p in r["poligonos"]]
    inter = pd.DataFrame(linhas, columns=["fid", "objectid", "cota_cm"])
    antes = len(inter)
    inter = inter.drop_duplicates()
    duplicatas_inter = antes - len(inter)
    if inter[["fid", "objectid"]].duplicated().any():
        raise ValueError("Mesmo poligono com cotas conflitantes para uma casa.")
    if inter["objectid"].map(lambda v: isinstance(v, bool)).any():
        raise ValueError("objectid booleano invalido.")
    inter["objectid"] = pd.to_numeric(inter["objectid"], errors="raise")
    if inter["objectid"].isna().any() or (inter["objectid"] % 1 != 0).any():
        raise ValueError("objectid deve ser inteiro e preenchido.")
    inter["objectid"] = inter["objectid"].astype("int64")
    if inter["cota_cm"].map(lambda v: isinstance(v, bool)).any():
        raise ValueError("Cota booleana invalida.")
    inter["cota_cm"] = pd.to_numeric(inter["cota_cm"], errors="raise")
    if not inter["cota_cm"].dropna().isin(COTAS_CM).all():
        raise ValueError("Cota fora da grade municipal documentada.")
    if inter.groupby("objectid")["cota_cm"].nunique(dropna=False).gt(1).any():
        raise ValueError("Um objectid esta associado a cotas diferentes.")
    if set(inter["cota_cm"].dropna()) != set(COTAS_CM):
        raise ValueError("O conjunto coletado nao comprova toda a grade de cotas.")
    inter["nivel_m"] = inter["cota_cm"] / 100
    inter = inter.merge(casas[["fid", "nome"]], on="fid", validate="many_to_one")
    inter = inter[["fid", "nome", "objectid", "cota_cm", "nivel_m"]].sort_values(["fid", "objectid"])

    # P7: resumo por casa. O merge a esquerda preserva a CB-1, sem poligonos.
    grupos = inter.groupby("fid").agg(
        qtd_poligonos=("objectid", "size"), menor_cota_cm=("cota_cm", "min"),
        qtd_cotas_ausentes=("cota_cm", lambda s: s.isna().sum()))
    resumo = casas.merge(grupos, on="fid", how="left", validate="one_to_one")
    # Zero e valido para quantidade de poligonos; NAO para a cota ausente.
    for coluna in ["qtd_poligonos", "qtd_cotas_ausentes"]:
        resumo[coluna] = resumo[coluna].fillna(0).astype(int)
    resumo["menor_cota_m"] = resumo["menor_cota_cm"] / 100
    ocorrencias = []
    lacunas_por_fid = {}
    for _, casa in resumo.iterrows():
        fid = int(casa["fid"])
        cotas = set(inter.loc[inter["fid"].eq(fid), "cota_cm"].dropna())
        minimo = min(cotas) if cotas else None
        lacunas = [c / 100 for c in COTAS_CM if minimo is not None and c >= minimo and c not in cotas]
        lacunas_por_fid[fid] = lacunas
        if casa["qtd_poligonos"] == 0:
            ocorrencias.append({"fid": fid, "nome": casa["nome"], "tipo": "sem_intersecao", "tratamento": "manter cota null; ausencia de intersecao nao comprova ausencia de risco"})
        if lacunas:
            ocorrencias.append({"fid": fid, "nome": casa["nome"], "tipo": "lacunas_apos_primeira_intersecao", "niveis_m": lacunas, "tratamento": "preservar e revisar com a fonte; nao interpolar"})
        if casa["qtd_cotas_ausentes"]:
            ocorrencias.append({"fid": fid, "nome": casa["nome"], "tipo": "cota_ausente_em_poligono", "tratamento": "manter ausente; menor cota considera apenas valores conhecidos"})
        salvo = unicas[fid].get("menor_cota_cm")
        if salvo != minimo:
            ocorrencias.append({"fid": fid, "nome": casa["nome"], "tipo": "minimo_recalculado", "antes": salvo, "depois": minimo})
    resumo["lacunas_apos_primeira_intersecao_m"] = resumo["fid"].map(lacunas_por_fid)

    # Grade observada: nao preencher lacunas usando nivel >= menor_cota_m.
    niveis = pd.DataFrame({"nivel_m": [c / 100 for c in COTAS_CM]})
    grade = casas[["fid", "nome"]].merge(niveis, how="cross")
    pares = set(inter[["fid", "nivel_m"]].dropna().itertuples(index=False, name=None))
    grade["intersecao_retornada"] = [(fid, nivel) in pares for fid, nivel in grade[["fid", "nivel_m"]].itertuples(index=False, name=None)]
    return resumo, inter, grade, ocorrencias, {
        "consultas_identicas_removidas": duplicatas_consultas,
        "intersecoes_identicas_removidas": duplicatas_inter,
    }


In [8]:
resumo, inter, grade, ocorrencias, duplicatas = clean_cotas(ler_json(caminho_cotas), casas)
print("Casas:", len(casas))
print("Intersecoes:", len(inter))
print("Combinacoes casa-nivel:", len(grade))
resumo


Casas: 24
Intersecoes: 442
Combinacoes casa-nivel: 600


,fid,nome,longitude,latitude,qtd_poligonos,menor_cota_cm,qtd_cotas_ausentes,menor_cota_m,lacunas_apos_primeira_intersecao_m
0,1,CB-2,-51.214362,-30.016007,12,430.0,0,4.3,[]
1,2,CB-3,-51.210352,-30.008697,19,350.0,0,3.5,[3.6]
2,3,CB-4,-51.207673,-30.002063,24,310.0,0,3.1,[]
3,4,CB-5,-51.201240,-29.982009,18,340.0,0,3.4,"[3.8, 3.9, 4.0]"
4,5,CB-6,-51.175796,-29.969781,12,430.0,0,4.3,[]
5,6,CB-9,-51.138591,-29.968089,25,300.0,0,3.0,[]
6,7,CB-11B,-51.242660,-30.092271,7,480.0,0,4.8,[]
7,8,CB-12,-51.236010,-30.061403,24,310.0,0,3.1,[]
8,9,CB-13,-51.232040,-30.054590,25,300.0,0,3.0,[]
9,10,CB-14,-51.215225,-30.048152,20,350.0,0,3.5,[]


In [9]:
pd.DataFrame(ocorrencias)


,fid,nome,tipo,niveis_m,tratamento
0,2,CB-3,lacunas_apos_primeira_intersecao,[3.6],preservar e revisar com a fonte; nao interpolar
1,4,CB-5,lacunas_apos_primeira_intersecao,"[3.8, 3.9, 4.0]",preservar e revisar com a fonte; nao interpolar
2,20,CB-1,sem_intersecao,NaN,manter cota null; ausencia de intersecao nao c...


In [10]:
nivel_escolhido = 3.60
if nivel_escolhido not in set(grade["nivel_m"]):
    raise ValueError("Nivel nao representado na coleta municipal.")
selecao = grade.loc[grade["nivel_m"].eq(nivel_escolhido)].copy()
print("Casas com intersecao retornada:", int(selecao["intersecao_retornada"].sum()))
selecao


Casas com intersecao retornada: 14


,fid,nome,nivel_m,intersecao_retornada
6,1,CB-2,3.6,False
31,2,CB-3,3.6,False
56,3,CB-4,3.6,True
81,4,CB-5,3.6,True
106,5,CB-6,3.6,False
131,6,CB-9,3.6,True
156,7,CB-11B,3.6,False
181,8,CB-12,3.6,True
206,9,CB-13,3.6,True
231,10,CB-14,3.6,True


In [11]:
def salvar_json(df, caminho):
    # pandas serializa valores ausentes como null, sem escrever NaN no JSON.
    df.to_json(caminho, orient="records", force_ascii=False, indent=2)


In [12]:
saida = raiz / "dados/processados"
relatorios = raiz / "relatorios"
saida.mkdir(parents=True, exist_ok=True)
relatorios.mkdir(parents=True, exist_ok=True)
# Nomes distintos dos arquivos do pacote anterior para facilitar a comparacao.
for nome, tabela in [("pipeline_casas_limpas.json", casas),
                     ("pipeline_cotas_resumo.json", resumo),
                     ("pipeline_intersecoes.json", inter),
                     ("pipeline_casas_por_nivel.json", grade)]:
    salvar_json(tabela, saida / nome)
auditoria.update({"escopo": "bases municipais; INCT ainda nao fornecido",
                  "entradas": [str(caminho_casas.relative_to(raiz)), str(caminho_cotas.relative_to(raiz))],
                  "quantidades": {"casas": len(casas), "intersecoes": len(inter), "grade": len(grade)},
                  "duplicatas_cotas": duplicatas, "ocorrencias": ocorrencias,
                  "data_coleta": None,
                  "limitacoes": ["Data exata de coleta nao registrada nas entradas.",
                                  "Faixa externa de 3,00 a 5,35 m; nao extrapolar ate 10 m.",
                                  "Menor cota e primeira intersecao retornada, nao medida de alagamento real ou falha da bomba.",
                                  "Resultados sem intersecao nao certificam seguranca.",
                                  "Coordenadas convertidas foram reutilizadas; projecao nao recalculada.",
                                  "Sem geometrias, nao e possivel refazer intersecoes ou validar topologia offline.",
                                  "Nao ha bairro nestas bases; 5,35 m nao integra a grade de 0,1 m do INCT."]})
(relatorios / "auditoria_pipeline.json").write_text(json.dumps(auditoria, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Limpeza concluida: {len(casas)} casas; {len(inter)} intersecoes; {len(grade)} combinacoes casa-nivel.")
print("Veja relatorios/auditoria_pipeline.json para conferir os tratamentos.")


Limpeza concluida: 24 casas; 442 intersecoes; 600 combinacoes casa-nivel.
Veja relatorios/auditoria_pipeline.json para conferir os tratamentos.
